# Convert Question to Solution Approach
This notebook has the code to use an LLM to find the solution for a question and convert that to a solution approach. Then, we can cluster questions based on the solution approach, rather than just similarity of questions.

In [1]:
%pip install openai
%pip install dotenv


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import asyncio
import os
import re
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
load_dotenv()  # read .env file
api_key = os.getenv("OPENAI_API_KEY")

In [25]:
filename = '../output/bin-th/bin-th-1.md'
questions = []
with open(filename, 'r') as f:
    questions = f.readlines()
print(f"read {len(questions)} questions from file {filename}")
all_questions = ''.join(questions[0:5])
print(f'All questions: {all_questions}')

read 12 questions from file ../output/bin-th/bin-th-1.md
All questions:    1. In the expansion of $\left(2^x+\dfrac{1}{4^x}\right)^n$, $n\in\mathbb{N}$, if sum of the coefficients of $2^{nd}$ and $3^{rd}$ term is $36$, then which of these are correct?   (1) $n=8$   (2) $n=9$   (3) $\dfrac{T_3}{T_2}=\dfrac{7}{4}$ when $x=-\dfrac{1}{3}$   (4) $\dfrac{T_3}{T_2}=7$ when $x=\dfrac{1}{3}$  
   2. The number of rational terms in the expansion of $\left(\sqrt{2}+3^{1/3}\right)^{100}$ is:   (1) 34   (2) 51   (3) 17   (4) 16  
   3. The co-efficient of $x^{160}$ in the expansion of $(x^8+1)^{60}\left(x^{12}+3x^4+\dfrac{3}{x^4}+\dfrac{1}{x^{12}}\right)^{-10}$ is:   (1) ${}^{30}C_5$   (2) ${}^{30}C_6$   (3) ${}^{30}C_{24}$   (4) ${}^{30}C_{26}$  
   4. $\displaystyle \sum_{j=1}^{m}\sum_{i=1}^{n} i\,j^2=$   (1) $\dfrac{mn(n+1)(m+1)(2n+1)}{12}$   (2) $\dfrac{n(n+1)m(m+1)(2m+1)}{12}$   (3) $\dfrac{mn(n+1)(m+1)}{4}$   (4) $\dfrac{mn(n+1)(m+1)(2m+1)}{6}$  
   5. If $(1+3x-2x^2)^{20}=a_0+a_1x+a_2x^2+\ld

In [22]:
# Read system prompt
system_prompt_file = '../src/solution_approach_prompt.md'
with open(system_prompt_file, 'r') as f:
    system_prompt_template = ''.join(f.readlines())
print(f"System prompt: {system_prompt_template}")

System prompt: # SYSTEM PROMPT
Role: Math Question Approach Extractor (Embedding-Optimized)

You are an expert high-school mathematics teacher and curriculum designer.

Your task is NOT to solve questions or compute answers.
Your task is to extract a normalized description of the solution approach so that questions can be clustered by similarity of method using sentence embeddings (bge-m3).

You must focus on HOW the question is solved, not on the final result.


## Core Objective

Given:
- a mathematics topic (e.g., Binomial Theorem), and
- one or more questions written using mathematical symbols,

analyze the dominant solution approach for each question and output a single standardized line per question describing that approach.

The output must be consistent, repeatable, and embedding-friendly.


## What to Extract (Per Question)

For each question, extract ONLY the following four fields:

1. SUB  
   The specific question type within the given topic
   (chosen from the topic-specif

In [15]:
def read_vocabulary(vocab_file_name):
    lines = []
    with open(vocab_file_name, 'r') as f:
        lines = f.readlines()
    vocabulary = "\n".join(str(x).strip() for x in lines if str(x).strip())
    return vocabulary

In [20]:
def read_all_vocabularies():
    vocabularies = {}
    for vocabulary_type in ['inputs', 'theorems', 'tech']:
        vocabulary = read_vocabulary(f'../src/taxonomies/global_canonical_{taxonomy_type}_vocabulary.md')
        vocabularies[vocabulary_type] = vocabulary
    vocabularies['sub'] = read_vocabulary(f'../src/taxonomies/binomial_theorem_vocabulary.md')
    return vocabularies

In [21]:
def inject_vocabulary(template, vocabularies):
    replacements = {
        "{{SUB_VOCAB}}": vocabularies['sub'],
        "{{INPUTS_VOCAB}}": vocabularies['inputs'],
        "{{THEOREMS_VOCAB}}": vocabularies['theorems'],
        "{{TECH_VOCAB}}": vocabularies['tech'],
    }
    out = system_prompt_template
    for placeholder, value in replacements.items():
        if placeholder not in out:
            raise ValueError(f"Placeholder missing in template: {placeholder}")
        out = out.replace(placeholder, value)
    return out

In [23]:
vocabularies = read_all_vocabularies()
system_prompt = inject_vocabulary(system_prompt_template, vocabularies)
print(f'System prompt: {system_prompt}')

System prompt: # SYSTEM PROMPT
Role: Math Question Approach Extractor (Embedding-Optimized)

You are an expert high-school mathematics teacher and curriculum designer.

Your task is NOT to solve questions or compute answers.
Your task is to extract a normalized description of the solution approach so that questions can be clustered by similarity of method using sentence embeddings (bge-m3).

You must focus on HOW the question is solved, not on the final result.


## Core Objective

Given:
- a mathematics topic (e.g., Binomial Theorem), and
- one or more questions written using mathematical symbols,

analyze the dominant solution approach for each question and output a single standardized line per question describing that approach.

The output must be consistent, repeatable, and embedding-friendly.


## What to Extract (Per Question)

For each question, extract ONLY the following four fields:

1. SUB  
   The specific question type within the given topic
   (chosen from the topic-specif

In [ ]:
client = OpenAI()
response = client.chat.completions.create (
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "system", 
                        "content": f"{system_prompt}"
                    },
                    {
                        "role": "user", 
                        "content": [
                            {
                                "type": "text",
                                "text": 
                                    f"""You will be given 5 questions from the topic BINOMIAL THEOREM.
                                    Produce exactly 5 output lines, one per question, in the same order.
                                    {all_questions}"""
                            }
                        ]
                    }
                ]
print(response.choices[0].message.content)

SyntaxError: closing parenthesis ']' does not match opening parenthesis '{' on line 9 (3646291603.py, line 20)